# 03.1 — Etiquetado local con LLM

Ejecuta la taxonomía peruana con un modelo local servido por LM Studio/llmster. El flujo usa JSON Schema, validación semántica, reintentos, guardado incremental y reanudación por `chunk_id`.

> Las celdas que llaman al modelo están desactivadas por defecto. Primero ejecuta el preflight y revisa los parámetros.

In [1]:
# Dependencias reproducibles del cuaderno.
from pathlib import Path
_req_candidates = [Path('requirements.txt'), Path('03_1_etiquetado_llm/requirements.txt')]
_requirements = next((p.resolve() for p in _req_candidates if p.exists()), None)
if _requirements is None:
    raise FileNotFoundError('No se encontró requirements.txt del módulo 03_1_etiquetado_llm')
%pip install -q -r {_requirements}

Note: you may need to restart the kernel to use updated packages.


## 1. Configuración

Usa como `PRIMARY_MODEL_ID` el identificador mostrado por `lms ps` o `/v1/models`. Para Qwen se propone `QW3`; para Gemma, `GEM`. Son identificadores constantes de tres caracteres compatibles con el frontend.

In [2]:
from __future__ import annotations

from collections import Counter, defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
import hashlib
import json
import math
import os
import random
import re
import subprocess
import sys
import time

import jsonschema
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm.auto import tqdm

def encontrar_raiz(inicio: Path | None = None) -> Path:
    inicio = (inicio or Path.cwd()).resolve()
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl').exists():
            return candidato
    raise FileNotFoundError('No se encontró la raíz del proyecto desde ' + str(inicio))

ROOT = encontrar_raiz()
MODULE_DIR = ROOT / '03_1_etiquetado_llm'
PROCESSED_DIR = ROOT / 'datos' / 'processed'
OUTPUT_DIR = ROOT / 'datos' / 'etiquetado' / 'llm_local'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_FILE = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
TAXONOMY_FILE = PROCESSED_DIR / 'taxonomia_moderacion.csv'
SKILL_FILE = ROOT / 'modelos' / 'skills' / 'clasificacion_moderacion_peru.md'
OPERATIVE_PROMPT_FILE = ROOT / 'para_equiquetado_LLM' / 'PROMPT_ETIQUETADO_LLM.md'
COMPACT_PROMPT_FILE = MODULE_DIR / 'prompt_operacional_compacto.md'
REFERENCE_GLOB = 'cgt_labeled_chunks_parte_*.jsonl'

API_BASE = os.getenv('LMSTUDIO_BASE_URL', 'http://localhost:1234/v1').rstrip('/')
PROMPT_MODE = 'compact'  # 'compact' para producción; 'full' solo para auditorías pequeñas
PRIMARY_MODEL_KEY = os.getenv('LMSTUDIO_PRIMARY_MODEL_KEY', 'qwen/qwen3.5-9b')
PRIMARY_MODEL_ID = os.getenv('LMSTUDIO_PRIMARY_MODEL', 'qwen-local-primary')
PRIMARY_ANNOTATOR_ID = 'QW3'
REVIEW_MODEL_ID = os.getenv('LMSTUDIO_REVIEW_MODEL', '')    # p. ej. gemma-local-review
REVIEW_ANNOTATOR_ID = 'GEM'

TEMPERATURE = 0.0
MAX_TOKENS_PER_RECORD = 512
MAX_TOKENS_OVERHEAD = 64
MAX_TOKENS_RETRY_MULTIPLIER = 2
REQUEST_TIMEOUT_SECONDS = 600
MODEL_CONTEXT_LENGTH = 16384
MODEL_LOAD_TIMEOUT_SECONDS = 600
MODEL_PARALLEL = 2
MAX_RETRIES = 2
MAX_WORKERS = 2
BATCH_SIZE = 2
PILOT_SIZE = 300
PILOT_SEED = 42
PILOT_LIMIT = None  # primero 5; usa None para completar los 300
SAFE_CONTROL_RATE = 0.10
LIVE_METRICS = True
PERSIST_METRICS = True

EJECUTAR_PILOTO = True     #False
EJECUTAR_PRODUCCION = False #True #False
EJECUTAR_REVISION = True   #False

print('Raíz:', ROOT)
print('Salidas:', OUTPUT_DIR)
print('API:', API_BASE)

Raíz: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4
Salidas: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_local
API: http://localhost:1234/v1


## 2. Preflight de archivos, daemon y modelos

El preflight no descarga ni etiqueta. Confirma que la autoridad normativa y la API están disponibles. Si LM Studio no responde, inicia automáticamente el daemon y el servidor. Si el modelo primario no está cargado con su identificador estable, ejecuta `lms load` y verifica que quede disponible antes de continuar.

In [3]:
ARCHIVOS_REQUERIDOS = [
    CHUNKS_FILE, TAXONOMY_FILE, SKILL_FILE, OPERATIVE_PROMPT_FILE, COMPACT_PROMPT_FILE
]
faltantes = [str(p) for p in ARCHIVOS_REQUERIDOS if not p.exists()]
if faltantes:
    raise FileNotFoundError('Faltan archivos requeridos:\n- ' + '\n- '.join(faltantes))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def consultar_modelos() -> list[str]:
    response = requests.get(f'{API_BASE}/models', timeout=10)
    response.raise_for_status()
    return [m['id'] for m in response.json().get('data', [])]

def ejecutar_lms(
    comando: list[str], timeout: int = 60, mostrar_salida: bool = True
) -> str:
    if mostrar_salida:
        print('Ejecutando:', ' '.join(comando))
    try:
        resultado = subprocess.run(
            comando, check=True, capture_output=True, text=True,
            encoding='utf-8', errors='replace', timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(
            'No se encontró el comando lms. Instala LM Studio y agrega su CLI al PATH.'
        ) from exc
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f'El comando {" ".join(comando)} excedió {timeout} segundos.') from exc
    except subprocess.CalledProcessError as exc:
        detalle = re.sub(
            r'\x1b\[[0-?]*[ -/]*[@-~]', '', exc.stderr or exc.stdout or str(exc)
        ).strip()
        raise RuntimeError(f'Falló {" ".join(comando)}: {detalle}') from exc
    salida = re.sub(
        r'\x1b\[[0-?]*[ -/]*[@-~]', '', resultado.stdout or resultado.stderr or ''
    ).strip()
    if salida and mostrar_salida:
        lineas = [
            linea for linea in salida.splitlines()
            if linea.strip() and not re.match(r'^Loading .+ \d+%', linea.strip())
        ]
        salida_visible = '\n'.join(lineas[-5:])
        encoding_salida = getattr(sys.stdout, 'encoding', None) or 'utf-8'
        print(
            salida_visible.encode(encoding_salida, errors='replace').decode(encoding_salida)
        )
    return salida

def iniciar_lmstudio() -> None:
    ejecutar_lms(['lms', 'daemon', 'up'])
    ejecutar_lms(['lms', 'server', 'start', '--port', '1234'])

def estado_modelo_primario() -> dict | None:
    salida = ejecutar_lms(['lms', 'ps', '--json'], mostrar_salida=False)
    try:
        modelos = json.loads(salida or '[]')
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'lms ps --json devolvió una salida inválida: {salida[:500]}') from exc
    return next((m for m in modelos if m.get('identifier') == PRIMARY_MODEL_ID), None)

def cargar_modelo_primario(recargar: bool = False) -> None:
    if not PRIMARY_MODEL_KEY:
        raise ValueError('Configura PRIMARY_MODEL_KEY con la clave mostrada por lms ls.')
    if not PRIMARY_MODEL_ID:
        raise ValueError('Configura PRIMARY_MODEL_ID antes de cargar el modelo.')
    if recargar:
        ejecutar_lms(['lms', 'unload', PRIMARY_MODEL_ID])
    ejecutar_lms(
        [
            'lms', 'load', PRIMARY_MODEL_KEY,
            '--context-length', str(MODEL_CONTEXT_LENGTH),
            '--gpu', 'off', '--parallel', str(MODEL_PARALLEL),
            '--identifier', PRIMARY_MODEL_ID, '-y',
        ],
        timeout=MODEL_LOAD_TIMEOUT_SECONDS,
    )

try:
    modelos_visibles = consultar_modelos()
except requests.RequestException:
    print('LM Studio no responde; se iniciarán el daemon y el servidor local.')
    iniciar_lmstudio()
    ultimo_error = None
    for _ in range(15):
        try:
            modelos_visibles = consultar_modelos()
            break
        except requests.RequestException as exc:
            ultimo_error = exc
            time.sleep(2)
    else:
        raise RuntimeError(
            f'LM Studio no respondió en {API_BASE} después de iniciar el servidor.'
        ) from ultimo_error

estado_primario = estado_modelo_primario()
configuracion_correcta = bool(
    estado_primario
    and estado_primario.get('modelKey') == PRIMARY_MODEL_KEY
    and estado_primario.get('contextLength') == MODEL_CONTEXT_LENGTH
    and estado_primario.get('parallel') == MODEL_PARALLEL
)
if not configuracion_correcta:
    print(
        f'El modelo {PRIMARY_MODEL_ID!r} no está cargado con la configuración óptima; '
        f'se cargará {PRIMARY_MODEL_KEY!r} con parallel={MODEL_PARALLEL}.'
    )
    cargar_modelo_primario(recargar=estado_primario is not None)
    for _ in range(15):
        modelos_visibles = consultar_modelos()
        if PRIMARY_MODEL_ID in modelos_visibles:
            break
        time.sleep(2)
    else:
        raise RuntimeError(
            f'El modelo se cargó, pero {PRIMARY_MODEL_ID!r} no apareció en {API_BASE}/models.'
        )
    estado_primario = estado_modelo_primario()
if not estado_primario or estado_primario.get('parallel') != MODEL_PARALLEL:
    raise RuntimeError('LM Studio no aplicó la configuración parallel solicitada.')

print('LM Studio responde correctamente.')
print('Modelos visibles:', modelos_visibles or '(ninguno)')
print('SHA256 skill:', sha256_file(SKILL_FILE))
print('SHA256 prompt:', sha256_file(OPERATIVE_PROMPT_FILE))
print('SHA256 prompt compacto:', sha256_file(COMPACT_PROMPT_FILE))
print('Modo de prompt:', PROMPT_MODE)
print(
    f'Modelo primario listo: {PRIMARY_MODEL_ID} ({PRIMARY_MODEL_KEY}), '
    f'parallel={MODEL_PARALLEL}, workers={MAX_WORKERS}, batch={BATCH_SIZE}'
)

LM Studio responde correctamente.
Modelos visibles: ['qwen-local-primary', 'qwen/qwen3.5-9b', 'text-embedding-nomic-embed-text-v1.5']
SHA256 skill: 45f9d3231a92453835ee6dfcbb8cfff0b682718caa4111f4bca3e841573a0efb
SHA256 prompt: 47f85a096736a35cf7499a084a5ca502ecb4c8af51a6a01249bbe2987cac433b
SHA256 prompt compacto: 9849a4eebec04428aef7df625f0a44e8317389cf50d43b4488164a56bae6d158
Modo de prompt: compact
Modelo primario listo: qwen-local-primary (qwen/qwen3.5-9b), parallel=2, workers=2, batch=2


## 3. Carga canónica y taxonomía

La entrada puede ocupar decenas de MB, pero cabe holgadamente en la RAM disponible. Se valida `chunk_id` antes de cualquier inferencia.

In [4]:
def leer_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for lineno, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'JSON inválido en {path}, línea {lineno}: {exc}') from exc
    return rows

taxonomy_df = pd.read_csv(TAXONOMY_FILE).fillna('')
ALLOWED_FLAGS = set(taxonomy_df.loc[taxonomy_df['categoria'] == 'FLAG', 'label'])
ALLOWED_LABELS = set(taxonomy_df.loc[taxonomy_df['categoria'] != 'FLAG', 'label'])
SAFE_LABELS = {'seguro', 'seguro_ironia_marcada'}
DAMAGE_LABELS = ALLOWED_LABELS - SAFE_LABELS
LABEL_ORDER = taxonomy_df.loc[taxonomy_df['categoria'] != 'FLAG', 'label'].tolist()
FLAG_ORDER = taxonomy_df.loc[taxonomy_df['categoria'] == 'FLAG', 'label'].tolist()

chunks = leer_jsonl(CHUNKS_FILE)
chunk_ids = [r.get('chunk_id') for r in chunks]
if any(not x for x in chunk_ids):
    raise ValueError('Hay chunk_id nulos o vacíos en el canónico.')
if len(chunk_ids) != len(set(chunk_ids)):
    raise ValueError('Hay chunk_id duplicados en el canónico.')
if any(not isinstance(r.get('text'), str) or not r['text'].strip() for r in chunks):
    raise ValueError('Hay chunks sin texto válido.')

CHUNK_BY_ID = {r['chunk_id']: r for r in chunks}
CANONICAL_POSITION = {r['chunk_id']: i for i, r in enumerate(chunks)}
print(f'Chunks: {len(chunks):,}')
print(f'Etiquetas: {len(ALLOWED_LABELS)} | Flags: {len(ALLOWED_FLAGS)}')

Chunks: 69,853
Etiquetas: 14 | Flags: 3


## 4. Prompt trazable y JSON Schema

El modo compacto conserva los criterios operativos y registra los hashes de las fuentes completas; el modo full las inserta literalmente. El envoltorio `annotations` permite procesar lotes y cada elemento se transforma después en una línea JSONL con campos administrativos constantes.

In [5]:
skill_text = SKILL_FILE.read_text(encoding='utf-8')
operative_prompt_text = OPERATIVE_PROMPT_FILE.read_text(encoding='utf-8')
compact_prompt_text = COMPACT_PROMPT_FILE.read_text(encoding='utf-8')
taxonomy_text = TAXONOMY_FILE.read_text(encoding='utf-8')
if PROMPT_MODE not in {'compact', 'full'}:
    raise ValueError("PROMPT_MODE debe ser 'compact' o 'full'.")
authority_text = (
    compact_prompt_text
    if PROMPT_MODE == 'compact'
    else f'=== SKILL ===\n{skill_text}\n\n=== PROMPT OPERATIVO ===\n{operative_prompt_text}'
)

SYSTEM_PROMPT = f'''Eres el clasificador local de este proyecto.
Las siguientes fuentes son la autoridad normativa completa. No uses una taxonomía externa.

=== REGLAS OPERATIVAS ({PROMPT_MODE}) ===
{authority_text}

=== TAXONOMÍA CSV ===
{taxonomy_text}

ADAPTACIÓN TÉCNICA LOCAL:
- Recibirás de 1 a {BATCH_SIZE} chunks por llamada.
- Devuelve el objeto raíz annotations exigido por el JSON Schema.
- Conserva exactamente el orden y chunk_id de entrada.
- Analiza cada chunk de forma independiente.
- No expongas razonamiento interno; justifica brevemente el criterio aplicado.
- El programa añadirá los campos administrativos y escribirá una línea JSONL por anotación.
'''

SEMANTIC_FIELDS = {
    'chunk_id', 'labels', 'flags', 'needs_review', 'notes',
    'score_confianza', 'justificacion'
}
FINAL_FIELDS = {
    'chunk_id', 'labels', 'flags', 'needs_review', 'notes',
    'annotator_type', 'annotator_id', 'annotator_model', 'skill_file',
    'score_confianza', 'justificacion', 'annotated_at'
}

def response_schema(batch_length: int) -> dict:
    annotation = {
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'chunk_id': {'type': 'string', 'minLength': 1, 'maxLength': 160},
            'labels': {
                'type': 'array', 'minItems': 1, 'uniqueItems': True,
                'items': {'type': 'string', 'enum': sorted(ALLOWED_LABELS)},
            },
            'flags': {
                'type': 'array', 'uniqueItems': True,
                'items': {'type': 'string', 'enum': sorted(ALLOWED_FLAGS)},
            },
            'needs_review': {'type': 'boolean'},
            'notes': {'type': 'string', 'maxLength': 160},
            'score_confianza': {'type': 'number', 'minimum': 0, 'maximum': 1},
            'justificacion': {'type': 'string', 'minLength': 1, 'maxLength': 500},
        },
        'required': sorted(SEMANTIC_FIELDS),
    }
    schema = {
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'annotations': {
                'type': 'array', 'minItems': batch_length, 'maxItems': batch_length,
                'items': annotation,
            }
        },
        'required': ['annotations'],
    }
    return {
        'type': 'json_schema',
        'json_schema': {'name': 'moderacion_peru_batch', 'strict': True, 'schema': schema},
    }

## 5. Cliente, reglas de validación y reintentos

In [6]:
def validar_semantica(row: dict, expected_id: str) -> list[str]:
    errors = []
    if set(row) != SEMANTIC_FIELDS:
        errors.append(f'campos inesperados/faltantes: {sorted(set(row) ^ SEMANTIC_FIELDS)}')
    if row.get('chunk_id') != expected_id:
        errors.append(f'chunk_id esperado {expected_id!r}, recibido {row.get("chunk_id")!r}')
    labels = row.get('labels', [])
    flags = row.get('flags', [])
    if not isinstance(labels, list) or not labels:
        errors.append('labels debe ser una lista no vacía')
    elif not set(labels) <= ALLOWED_LABELS:
        errors.append('labels contiene valores fuera de la taxonomía')
    if not isinstance(flags, list) or not set(flags) <= ALLOWED_FLAGS:
        errors.append('flags contiene valores fuera de la taxonomía')
    safe = set(labels) & SAFE_LABELS
    damage = set(labels) & DAMAGE_LABELS
    if safe and damage:
        errors.append('una etiqueta segura no puede coexistir con daño')
    if len(safe) > 1:
        errors.append('seguro y seguro_ironia_marcada no deben coexistir')
    score = row.get('score_confianza')
    if isinstance(score, bool) or not isinstance(score, (int, float)) or not 0 <= score <= 1:
        errors.append('score_confianza debe estar entre 0 y 1')
    else:
        if ({'ironia_ambigua', 'contexto_necesario'} & set(flags)) and score > 0.65:
            errors.append('flag ambiguo/contextual limita score_confianza a 0.65')
        if (flags or score < 0.70) and row.get('needs_review') is not True:
            errors.append('flags o score < 0.70 obligan needs_review=true')
    if not isinstance(row.get('needs_review'), bool):
        errors.append('needs_review debe ser booleano')
    if not isinstance(row.get('notes'), str):
        errors.append('notes debe ser texto')
    if not isinstance(row.get('justificacion'), str) or not row.get('justificacion', '').strip():
        errors.append('justificacion no puede estar vacía')
    return errors

def completar_fila(row: dict, model_id: str, annotator_id: str) -> dict:
    return {
        'chunk_id': row['chunk_id'],
        'labels': row['labels'],
        'flags': row['flags'],
        'needs_review': row['needs_review'],
        'notes': row['notes'],
        'annotator_type': 'llm',
        'annotator_id': annotator_id,
        'annotator_model': model_id,
        'skill_file': SKILL_FILE.name,
        'score_confianza': float(row['score_confianza']),
        'justificacion': row['justificacion'].strip(),
        'annotated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
    }

def construir_entrada(records: list[dict]) -> str:
    payload = []
    for r in records:
        item = {
            'chunk_id': r['chunk_id'],
            'text': r['text'],
            'channel_title': r.get('channel_title'),
            'video_title': r.get('video_title'),
        }
        for key in ('contexto_anterior', 'contexto_posterior'):
            if r.get(key):
                item[key] = r[key]
        payload.append(item)
    return 'Clasifica estos registros según las fuentes de autoridad:\n' + json.dumps(payload, ensure_ascii=False)

def llamar_lmstudio(
    records: list[dict], model_id: str, correction: str = '', max_tokens: int | None = None
) -> tuple[list[dict], dict]:
    user_content = construir_entrada(records)
    if correction:
        user_content += '\nLa respuesta anterior fue inválida. Corrige estos errores:\n' + correction
    token_budget = max_tokens or (
        MAX_TOKENS_OVERHEAD + MAX_TOKENS_PER_RECORD * len(records)
    )
    body = {
        'model': model_id,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_content},
        ],
        'temperature': TEMPERATURE,
        'max_tokens': token_budget,
        'stream': False,
        'response_format': response_schema(len(records)),
    }
    response = requests.post(
        f'{API_BASE}/chat/completions', json=body, timeout=REQUEST_TIMEOUT_SECONDS
    )
    response.raise_for_status()
    response_json = response.json()
    choice = response_json['choices'][0]
    reasoning_tokens = (
        response_json.get('usage', {}).get('completion_tokens_details', {}).get('reasoning_tokens', 0)
    )
    if reasoning_tokens:
        raise RuntimeError(
            f'El modelo usó {reasoning_tokens} tokens de razonamiento. ' 
            'Configura enableThinking=false y recarga el modelo.'
        )
    if choice.get('finish_reason') == 'length':
        raise RuntimeError(
            f'La respuesta agotó el límite de {token_budget} tokens antes de cerrar el JSON.'
        )
    content = choice['message']['content']
    if not content:
        raise RuntimeError('LM Studio devolvió content vacío.')
    parsed = content if isinstance(content, dict) else json.loads(content)
    jsonschema.validate(parsed, response_schema(len(records))['json_schema']['schema'])
    return parsed['annotations'], response_json.get('usage', {})

def clasificar_lote(records: list[dict], model_id: str, annotator_id: str) -> tuple[list[dict], dict]:
    if not model_id:
        raise ValueError('Debes configurar el identificador del modelo.')
    if not re.fullmatch(r'[A-Z0-9]{3}', annotator_id):
        raise ValueError('annotator_id debe tener exactamente tres caracteres A-Z/0-9.')
    correction = ''
    last_error = None
    base_token_budget = MAX_TOKENS_OVERHEAD + MAX_TOKENS_PER_RECORD * len(records)
    for attempt in range(1, MAX_RETRIES + 2):
        token_budget = base_token_budget * (MAX_TOKENS_RETRY_MULTIPLIER ** (attempt - 1))
        try:
            annotations, usage = llamar_lmstudio(
                records, model_id, correction, max_tokens=token_budget
            )
            expected = [r['chunk_id'] for r in records]
            received = [r.get('chunk_id') for r in annotations]
            if received != expected:
                raise ValueError(f'orden/IDs incorrectos: esperado={expected}, recibido={received}')
            all_errors = []
            for expected_id, row in zip(expected, annotations):
                all_errors.extend(f'{expected_id}: {e}' for e in validar_semantica(row, expected_id))
            if all_errors:
                raise ValueError('; '.join(all_errors))
            return [completar_fila(r, model_id, annotator_id) for r in annotations], usage
        except Exception as exc:
            last_error = exc
            correction = str(exc)[:3000]
            print(f'Intento {attempt}/{MAX_RETRIES + 1} inválido: {correction[:500]}')
            if attempt <= MAX_RETRIES:
                time.sleep(min(2 ** (attempt - 1), 8))
    raise RuntimeError(f'El lote falló después de {MAX_RETRIES + 1} intentos: {last_error}') from last_error

## 6. Muestra piloto reproducible

Incluye primero los IDs de la referencia existente y completa la muestra mediante recorrido balanceado por canal. Esto sirve para comparar modelos; no asigna etiquetas mediante heurísticas.

In [7]:
reference_files = sorted((ROOT / 'para_equiquetado_LLM').glob(REFERENCE_GLOB))
reference_rows = [row for path in reference_files for row in leer_jsonl(path)]
REFERENCE_BY_ID = {r['chunk_id']: r for r in reference_rows if r.get('chunk_id') in CHUNK_BY_ID}

def seleccionar_piloto(records: list[dict], n: int, seed: int, priority_ids: set[str]) -> list[dict]:
    n = min(n, len(records))
    selected_ids = []
    selected_set = set()
    for r in records:
        if r['chunk_id'] in priority_ids and len(selected_ids) < n:
            selected_ids.append(r['chunk_id'])
            selected_set.add(r['chunk_id'])
    groups = defaultdict(list)
    for r in records:
        if r['chunk_id'] not in selected_set:
            groups[r.get('channel_title') or '__sin_canal__'].append(r['chunk_id'])
    rng = random.Random(seed)
    for ids in groups.values():
        rng.shuffle(ids)
    channels = sorted(groups)
    while len(selected_ids) < n and channels:
        next_channels = []
        for channel in channels:
            if groups[channel] and len(selected_ids) < n:
                cid = groups[channel].pop()
                selected_ids.append(cid)
                selected_set.add(cid)
            if groups[channel]:
                next_channels.append(channel)
        channels = next_channels
    selected_ids.sort(key=CANONICAL_POSITION.get)
    return [CHUNK_BY_ID[cid] for cid in selected_ids]

pilot_rows = seleccionar_piloto(chunks, PILOT_SIZE, PILOT_SEED, set(REFERENCE_BY_ID))
pilot_manifest = OUTPUT_DIR / f'piloto_ids_n{len(pilot_rows)}_seed{PILOT_SEED}.json'
manifest = {
    'seed': PILOT_SEED,
    'n': len(pilot_rows),
    'canonical_sha256': sha256_file(CHUNKS_FILE),
    'skill_sha256': sha256_file(SKILL_FILE),
    'operative_prompt_sha256': sha256_file(OPERATIVE_PROMPT_FILE),
    'compact_prompt_sha256': sha256_file(COMPACT_PROMPT_FILE),
    'prompt_mode': PROMPT_MODE,
    'chunk_ids': [r['chunk_id'] for r in pilot_rows],
}
pilot_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Piloto: {len(pilot_rows)} chunks; referencia coincidente: {len(REFERENCE_BY_ID)}')
print('Manifiesto:', pilot_manifest)

Piloto: 300 chunks; referencia coincidente: 60
Manifiesto: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_local\piloto_ids_n300_seed42.json


## 7. Ejecutor incremental y reanudable

No reintenta IDs ya guardados después de validar el archivo existente. Ejecuta como máximo dos solicitudes concurrentes, pero valida y escribe sus resultados desde el hilo principal, en orden. Cada lote se fuerza a disco y actualiza un panel de métricas y un archivo lateral `*.metrics.json`.

In [8]:
def slug_model(model_id: str) -> str:
    return re.sub(r'[^a-zA-Z0-9._-]+', '_', model_id).strip('_').lower() or 'modelo_sin_id'

def validar_fila_final(row: dict, canonical_ids: set[str]) -> list[str]:
    errors = []
    if set(row) != FINAL_FIELDS:
        errors.append(f'campos inesperados/faltantes: {sorted(set(row) ^ FINAL_FIELDS)}')
    cid = row.get('chunk_id')
    if cid not in canonical_ids:
        errors.append(f'chunk_id ajeno al canónico: {cid!r}')
    semantic = {key: row.get(key) for key in SEMANTIC_FIELDS}
    errors.extend(validar_semantica(semantic, cid))
    if row.get('annotator_type') != 'llm':
        errors.append('annotator_type debe ser llm')
    if row.get('skill_file') != SKILL_FILE.name:
        errors.append('skill_file incorrecto')
    return errors

def cargar_progreso(path: Path, allowed_ids: set[str]) -> tuple[list[dict], set[str]]:
    if not path.exists():
        return [], set()
    existing = leer_jsonl(path)
    ids = [r.get('chunk_id') for r in existing]
    if len(ids) != len(set(ids)):
        raise ValueError(f'Hay chunk_id duplicados en {path}. No se reanudará.')
    for i, row in enumerate(existing, 1):
        errors = validar_fila_final(row, allowed_ids)
        if errors:
            raise ValueError(f'Fila existente inválida {i} en {path}: {errors}')
    return existing, set(ids)

def acumular_metricas(state: dict, rows: list[dict]) -> None:
    for row in rows:
        labels = set(row['labels'])
        flags = set(row['flags'])
        state['label_counts'].update(labels)
        state['flag_counts'].update(flags)
        state['completed'] += 1
        state['safe_chunks'] += int(bool(labels & SAFE_LABELS))
        state['damage_chunks'] += int(bool(labels & DAMAGE_LABELS))
        state['needs_review'] += int(bool(row['needs_review']))
        state['confidence_sum'] += float(row['score_confianza'])

def crear_estado_metricas(existing_rows: list[dict]) -> dict:
    state = {
        'label_counts': Counter(),
        'flag_counts': Counter(),
        'completed': 0,
        'safe_chunks': 0,
        'damage_chunks': 0,
        'needs_review': 0,
        'confidence_sum': 0.0,
    }
    acumular_metricas(state, existing_rows)
    return state

def resumen_metricas(state: dict, total_target: int) -> dict:
    completed = state['completed']
    return {
        'completed': completed,
        'pending': max(total_target - completed, 0),
        'progress_pct': round(100 * completed / total_target, 3) if total_target else 100.0,
        'safe_chunks': state['safe_chunks'],
        'damage_chunks': state['damage_chunks'],
        'needs_review': state['needs_review'],
        'needs_review_pct': round(100 * state['needs_review'] / completed, 3) if completed else 0.0,
        'mean_confidence': round(state['confidence_sum'] / completed, 4) if completed else None,
        'label_counts': {label: int(state['label_counts'][label]) for label in LABEL_ORDER},
        'flag_counts': {flag: int(state['flag_counts'][flag]) for flag in FLAG_ORDER},
    }

def tabla_metricas(state: dict, total_target: int) -> pd.DataFrame:
    summary = resumen_metricas(state, total_target)
    completed = summary['completed']
    rows = [
        {'grupo': 'PROGRESO', 'metrica': 'completados', 'valor': completed, 'porcentaje': summary['progress_pct']},
        {'grupo': 'PROGRESO', 'metrica': 'pendientes', 'valor': summary['pending'], 'porcentaje': round(100 - summary['progress_pct'], 3)},
        {'grupo': 'CLASIFICACION', 'metrica': 'chunks_seguro', 'valor': summary['safe_chunks'], 'porcentaje': round(100 * summary['safe_chunks'] / completed, 3) if completed else 0.0},
        {'grupo': 'CLASIFICACION', 'metrica': 'chunks_con_dano', 'valor': summary['damage_chunks'], 'porcentaje': round(100 * summary['damage_chunks'] / completed, 3) if completed else 0.0},
        {'grupo': 'CALIDAD', 'metrica': 'needs_review', 'valor': summary['needs_review'], 'porcentaje': summary['needs_review_pct']},
        {'grupo': 'CALIDAD', 'metrica': 'confianza_media', 'valor': summary['mean_confidence'], 'porcentaje': None},
    ]
    for label in LABEL_ORDER:
        count = summary['label_counts'][label]
        rows.append({
            'grupo': 'ETIQUETA', 'metrica': label, 'valor': count,
            'porcentaje': round(100 * count / completed, 3) if completed else 0.0,
        })
    for flag in FLAG_ORDER:
        count = summary['flag_counts'][flag]
        rows.append({
            'grupo': 'FLAG', 'metrica': flag, 'valor': count,
            'porcentaje': round(100 * count / completed, 3) if completed else 0.0,
        })
    return pd.DataFrame(rows)

def guardar_metricas(
    output_path: Path, state: dict, total_target: int, model_id: str, annotator_id: str,
) -> Path:
    metrics_path = output_path.with_suffix('.metrics.json')
    payload = {
        'source_output': str(output_path),
        'model': model_id,
        'annotator_id': annotator_id,
        'prompt_mode': PROMPT_MODE,
        'updated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
        **resumen_metricas(state, total_target),
    }
    temp_path = metrics_path.with_suffix(metrics_path.suffix + '.tmp')
    temp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temp_path, metrics_path)
    return metrics_path

def actualizar_panel_metricas(handle, state: dict, total_target: int):
    if not LIVE_METRICS:
        return handle
    table = tabla_metricas(state, total_target)
    if handle is None:
        return display(table, display_id=True)
    handle.update(table)
    return handle

def ejecutar_etiquetado(
    records: list[dict], output_path: Path, model_id: str, annotator_id: str,
    batch_size: int = BATCH_SIZE, limit: int | None = None,
    max_workers: int = MAX_WORKERS,
) -> dict:
    if batch_size < 1 or max_workers < 1:
        raise ValueError('batch_size y max_workers deben ser mayores que cero.')
    allowed_ids = {r['chunk_id'] for r in records}
    existing, completed_ids = cargar_progreso(output_path, allowed_ids)
    total_target = len(records)
    metrics_state = crear_estado_metricas(existing)
    metrics_handle = actualizar_panel_metricas(None, metrics_state, total_target)
    metrics_path = None
    if PERSIST_METRICS:
        metrics_path = guardar_metricas(
            output_path, metrics_state, total_target, model_id, annotator_id
        )
    pending = [r for r in records if r['chunk_id'] not in completed_ids]
    if limit is not None:
        pending = pending[:limit]
    if not pending:
        print('No hay registros pendientes para esta corrida.')
        return {
            'new_rows': 0, 'elapsed_seconds': 0, 'chunks_per_minute': None,
            'metrics_file': str(metrics_path) if metrics_path else None,
            **resumen_metricas(metrics_state, total_target),
        }
    started = time.perf_counter()
    usage_total = defaultdict(int)
    new_count = 0
    batches = (
        pending[start:start + batch_size]
        for start in range(0, len(pending), batch_size)
    )
    total_batches = math.ceil(len(pending) / batch_size)
    with (
        output_path.open('a', encoding='utf-8', newline='\n') as f,
        ThreadPoolExecutor(max_workers=max_workers) as executor,
        tqdm(total=total_batches, desc=output_path.stem) as progress,
    ):
        pendientes = deque()
        for _ in range(min(max_workers, total_batches)):
            batch = next(batches)
            future = executor.submit(clasificar_lote, batch, model_id, annotator_id)
            pendientes.append((batch, future))
        while pendientes:
            batch, future = pendientes.popleft()
            rows, usage = future.result()
            for row in rows:
                errors = validar_fila_final(row, allowed_ids)
                if errors:
                    raise ValueError(f'Salida final inválida para {row.get("chunk_id")}: {errors}')
                f.write(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n')
                new_count += 1
            f.flush()
            os.fsync(f.fileno())
            acumular_metricas(metrics_state, rows)
            if PERSIST_METRICS:
                metrics_path = guardar_metricas(
                    output_path, metrics_state, total_target, model_id, annotator_id
                )
            metrics_handle = actualizar_panel_metricas(
                metrics_handle, metrics_state, total_target
            )
            live = resumen_metricas(metrics_state, total_target)
            progress.update(1)
            progress.set_postfix(
                completados=live['completed'], dano=live['damage_chunks'],
                revision=live['needs_review'], confianza=live['mean_confidence'],
            )
            for key, value in usage.items():
                if isinstance(value, (int, float)):
                    usage_total[key] += value
            try:
                siguiente = next(batches)
            except StopIteration:
                continue
            future = executor.submit(
                clasificar_lote, siguiente, model_id, annotator_id
            )
            pendientes.append((siguiente, future))
    elapsed = time.perf_counter() - started
    stats = {
        'output': str(output_path),
        'model': model_id,
        'new_rows': new_count,
        'total_rows': len(existing) + new_count,
        'elapsed_seconds': round(elapsed, 2),
        'chunks_per_minute': round(new_count / elapsed * 60, 3) if elapsed else None,
        'usage': dict(usage_total),
        'metrics_file': str(metrics_path) if metrics_path else None,
        **resumen_metricas(metrics_state, total_target),
    }
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    return stats

## 8. Ejecutar piloto

La evaluación local seleccionó `BATCH_SIZE=2`, `MAX_WORKERS=2` y `MODEL_PARALLEL=2`: alcanzó 3.547 chunks/min y 4/4 coincidencias en la muestra medida. Usa `PILOT_LIMIT=5` para una prueba corta o `PILOT_LIMIT=None` para completar la muestra de 300. Para comparar otro modelo, cambia modelo/anotador: se generará otro archivo.

In [9]:
pilot_output = OUTPUT_DIR / f'{slug_model(PRIMARY_MODEL_ID)}_piloto.jsonl'
if EJECUTAR_PILOTO:
    pilot_stats = ejecutar_etiquetado(
        pilot_rows, pilot_output, PRIMARY_MODEL_ID, PRIMARY_ANNOTATOR_ID,
        batch_size=BATCH_SIZE, limit=PILOT_LIMIT,
    )
else:
    print('Piloto desactivado. Revisa parámetros y cambia EJECUTAR_PILOTO=True.')
print('Salida prevista:', pilot_output)

,grupo,metrica,valor,porcentaje
0,PROGRESO,completados,300.0000,100.000
1,PROGRESO,pendientes,0.0000,0.000
2,CLASIFICACION,chunks_seguro,243.0000,81.000
3,CLASIFICACION,chunks_con_dano,57.0000,19.000
4,CALIDAD,needs_review,21.0000,7.000
5,CALIDAD,confianza_media,0.9499,NaN
6,ETIQUETA,seguro,243.0000,81.000
7,ETIQUETA,seguro_ironia_marcada,0.0000,0.000
8,ETIQUETA,racismo_etnico_explicito,19.0000,6.333
9,ETIQUETA,racismo_linguistico,11.0000,3.667


qwen-local-primary_piloto:   0%|          | 0/146 [00:00<?, ?it/s]

Intento 1/3 inválido: CbAS71o-na0_0016: flag ambiguo/contextual limita score_confianza a 0.65
{
  "output": "D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\llm_local\\qwen-local-primary_piloto.jsonl",
  "model": "qwen-local-primary",
  "new_rows": 291,
  "total_rows": 300,
  "elapsed_seconds": 5634.97,
  "chunks_per_minute": 3.099,
  "usage": {
    "prompt_tokens": 481681,
    "completion_tokens": 42113,
    "total_tokens": 523794
  },
  "metrics_file": "D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\llm_local\\qwen-local-primary_piloto.metrics.json",
  "completed": 300,
  "pending": 0,
  "progress_pct": 100.0,
  "safe_chunks": 243,
  "damage_chunks": 57,
  "needs_review": 21,
  "needs_review_pct": 7.0,
  "mean_confidence": 0.9499,
  "label_counts": {
    "seguro": 243,
    "seguro_ironia_marcada": 0,
    "racismo_etnico_explicito": 19,
    "racismo_linguistico": 11,
    "clasismo_racial": 3,
    "discriminacion_regional": 14,
    "racismo_encubierto": 28,
   

## 9. Evaluación contra referencia

La referencia previa no sustituye un gold humano. Estas métricas sirven para comparar modelos bajo condiciones idénticas.

In [10]:
def evaluar_contra_referencia(prediction_file: Path, reference_by_id: dict[str, dict]) -> tuple[pd.DataFrame, dict]:
    predictions = leer_jsonl(prediction_file)
    pred_by_id = {r['chunk_id']: r for r in predictions}
    overlap_ids = [cid for cid in reference_by_id if cid in pred_by_id]
    if not overlap_ids:
        raise ValueError('No hay IDs compartidos con la referencia.')
    classes = sorted(ALLOWED_LABELS)
    mlb = MultiLabelBinarizer(classes=classes)
    mlb.fit([classes])
    y_true = mlb.transform([reference_by_id[cid]['labels'] for cid in overlap_ids])
    y_pred = mlb.transform([pred_by_id[cid]['labels'] for cid in overlap_ids])
    exact = float(np.mean([
        set(reference_by_id[cid]['labels']) == set(pred_by_id[cid]['labels']) for cid in overlap_ids
    ]))
    jaccard = []
    for cid in overlap_ids:
        a, b = set(reference_by_id[cid]['labels']), set(pred_by_id[cid]['labels'])
        jaccard.append(len(a & b) / len(a | b))
    report = classification_report(
        y_true, y_pred, target_names=classes, zero_division=0, output_dict=True
    )
    label_rows = []
    for label in classes:
        item = report[label]
        label_rows.append({
            'label': label, 'precision': item['precision'], 'recall': item['recall'],
            'f1': item['f1-score'], 'support': int(item['support']),
        })
    summary = {
        'n_overlap': len(overlap_ids),
        'exact_match': exact,
        'jaccard_mean': float(np.mean(jaccard)),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }
    return pd.DataFrame(label_rows), summary

if pilot_output.exists() and REFERENCE_BY_ID:
    per_label, evaluation_summary = evaluar_contra_referencia(pilot_output, REFERENCE_BY_ID)
    display(pd.DataFrame([evaluation_summary]))
    display(per_label.sort_values(['support', 'label'], ascending=[False, True]))
else:
    print('Ejecuta el piloto para calcular métricas.')

,n_overlap,exact_match,jaccard_mean,f1_macro,precision_macro,recall_macro
0,60,0.683333,0.683333,0.060383,0.063665,0.057423


,label,precision,recall,f1,support
9,seguro,0.891304,0.803922,0.845361,51
4,homofobia_transfobia,0.000000,0.000000,0.000000,4
11,sexual_cosificacion,0.000000,0.000000,0.000000,4
0,acoso_personal,0.000000,0.000000,0.000000,3
5,misoginia_acoso_genero,0.000000,0.000000,0.000000,1
1,amenaza_directa,0.000000,0.000000,0.000000,0
2,clasismo_racial,0.000000,0.000000,0.000000,0
3,discriminacion_regional,0.000000,0.000000,0.000000,0
6,racismo_encubierto,0.000000,0.000000,0.000000,0
7,racismo_etnico_explicito,0.000000,0.000000,0.000000,0


## 10. Producción completa

Activa esta celda solo después de elegir el modelo con el piloto. El archivo es independiente del piloto y reanuda sin volver a procesar IDs íntegros ya guardados.

In [11]:
production_output = OUTPUT_DIR / f'{slug_model(PRIMARY_MODEL_ID)}_labeled_chunks.jsonl'
if EJECUTAR_PRODUCCION:
    production_stats = ejecutar_etiquetado(
        chunks, production_output, PRIMARY_MODEL_ID, PRIMARY_ANNOTATOR_ID,
        batch_size=BATCH_SIZE, limit=None,
    )
else:
    print('Producción desactivada. Requiere EJECUTAR_PRODUCCION=True.')
print('Salida prevista:', production_output)

Producción desactivada. Requiere EJECUTAR_PRODUCCION=True.
Salida prevista: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\llm_local\qwen-local-primary_labeled_chunks.jsonl


## 11. Segunda pasada independiente con contexto

Selecciona todos los casos con revisión, daño o una muestra aleatoria de seguros. El segundo modelo no ve la etiqueta previa para evitar anclaje, pero sí recibe el chunk anterior y posterior del mismo video cuando existen. Su salida se conserva como otra anotación, sin sobrescribir la primera.

In [12]:
def construir_contexto_vecino(records: list[dict]) -> dict[str, dict]:
    by_video = defaultdict(list)
    for r in records:
        by_video[r.get('video_id')].append(r)
    enriched = {}
    for video_rows in by_video.values():
        video_rows.sort(key=lambda r: (r.get('start_seconds') or 0, CANONICAL_POSITION[r['chunk_id']]))
        for i, r in enumerate(video_rows):
            item = dict(r)
            if i > 0:
                item['contexto_anterior'] = video_rows[i - 1]['text']
            if i + 1 < len(video_rows):
                item['contexto_posterior'] = video_rows[i + 1]['text']
            enriched[r['chunk_id']] = item
    return enriched

def seleccionar_revision(primary_rows: list[dict], safe_rate: float, seed: int) -> list[str]:
    rng = random.Random(seed)
    selected = []
    for row in primary_rows:
        has_damage = bool(set(row['labels']) & DAMAGE_LABELS)
        if row['needs_review'] or has_damage:
            selected.append(row['chunk_id'])
        elif set(row['labels']) <= SAFE_LABELS and rng.random() < safe_rate:
            selected.append(row['chunk_id'])
    return selected

review_output = OUTPUT_DIR / f'{slug_model(REVIEW_MODEL_ID)}_revision.jsonl'
if EJECUTAR_REVISION:
    if not production_output.exists():
        raise FileNotFoundError('No existe la primera pasada de producción.')
    primary_rows = leer_jsonl(production_output)
    review_ids = seleccionar_revision(primary_rows, SAFE_CONTROL_RATE, PILOT_SEED)
    enriched = construir_contexto_vecino(chunks)
    review_records = [enriched[cid] for cid in review_ids]
    review_stats = ejecutar_etiquetado(
        review_records, review_output, REVIEW_MODEL_ID, REVIEW_ANNOTATOR_ID,
        batch_size=BATCH_SIZE, limit=None,
    )
else:
    print('Segunda pasada desactivada. Requiere EJECUTAR_REVISION=True.')
print('Salida prevista:', review_output)

FileNotFoundError: No existe la primera pasada de producción.

## 12. Auditoría final de una salida

In [ ]:
def auditar_salida(path: Path) -> dict:
    rows = leer_jsonl(path)
    ids = [r.get('chunk_id') for r in rows]
    errors = []
    if len(ids) != len(set(ids)):
        errors.append('chunk_id duplicados')
    for i, row in enumerate(rows, 1):
        row_errors = validar_fila_final(row, set(CHUNK_BY_ID))
        errors.extend(f'fila {i}: {e}' for e in row_errors)
    positions = [CANONICAL_POSITION[cid] for cid in ids if cid in CANONICAL_POSITION]
    if positions != sorted(positions):
        errors.append('el orden no coincide con el canónico')
    summary = {
        'path': str(path),
        'rows': len(rows),
        'unique_ids': len(set(ids)),
        'needs_review': sum(bool(r.get('needs_review')) for r in rows),
        'errors': errors[:50],
        'valid': not errors,
    }
    return summary

archivo_a_auditar = production_output if production_output.exists() else pilot_output
if archivo_a_auditar.exists():
    print(json.dumps(auditar_salida(archivo_a_auditar), ensure_ascii=False, indent=2))
else:
    print('Todavía no existe una salida para auditar.')